# 🔐 Notebook 6 — Authorization and Persona Readiness

This notebook validates the authorization model for the two Product Finder personas:

| Persona | Access |
|---------|--------|
| `external_customer` | Product recommendation, limited compatibility (with disclaimer), sample request |
| `internal_scientist` | Full recommendation, full compatibility + extended analysis, no sample restriction |

## How persona resolution works in this workshop
```
Client Request
    │  headers: Ocp-Apim-Subscription-Key + x-user-persona
    ▼
APIM Product Finder API
    │  Policy: validate sub key, pass x-user-persona downstream
    ▼
Orchestrator Agent
    │  Reads persona from message [GOVERNANCE CONTEXT] block
    │  Applies routing rules based on persona + intent + risk_tier
    ▼
Specialist Agents (capability-gated per persona)
```

> **Workshop mode (this lab):** no Entra app registration steps are required.
> This flow is designed to run with subscription Owner rights only, using APIM subscription key + persona simulation.

> **Production note:** In production, persona should be resolved from validated JWT claims
> (e.g., Entra ID app roles or group membership).

## What this notebook validates
1. APIM gateway and persona simulation flow are ready
2. APIM endpoint responds to a health/echo request
3. Persona matrix is documented and understood
4. Persona simulation mechanism is explained and ready

> Note: Agent-access APIM subscription key is created in Notebook 5 after agent APIs are available in APIM.

In [ ]:
import json, sys, pathlib, subprocess

def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "shared" / "utils.py").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing shared/utils.py")

REPO_ROOT = find_repo_root(pathlib.Path.cwd())
sys.path.insert(0, str(REPO_ROOT / "shared"))
import utils  # type: ignore[reportMissingImports]

def run(cmd, ok="", fail=""):
    return utils.run(cmd, ok, fail)

def azd_get(key: str) -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Missing azd env value [{key}]: {(p.stderr or p.stdout).strip()}")
    return (p.stdout or "").strip()

def azd_get_optional(key: str, default: str = "") -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        return default
    return (p.stdout or "").strip() or default

def set_azd_env(key: str, value: str):
    p = subprocess.run(["azd", "env", "set", key, value], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Failed to persist azd env {key}: {(p.stderr or p.stdout).strip()}")

hub_rg = azd_get("AZURE_RESOURCE_GROUP")
apim_out = run(
    f"az resource list -g {hub_rg} --resource-type Microsoft.ApiManagement/service -o json",
    "APIM query OK",
    "APIM query failed",
)
if not apim_out.success or not apim_out.json_data:
    raise RuntimeError("No APIM resource found in hub resource group.")
apim_name = apim_out.json_data[0]["name"]

PF_SUB_KEY = azd_get_optional("PF_AGENT_SUBSCRIPTION_KEY", "") or azd_get_optional("PF_SUBSCRIPTION_KEY", "")

utils.print_info(f"APIM:          {apim_name}")
utils.print_info(f"Sub key:       {PF_SUB_KEY[:8]}..." if PF_SUB_KEY else "Sub key: NOT FOUND (run Notebook 5 first)")

### 1️⃣ Validate APIM subscription key is present

In [ ]:
if not PF_SUB_KEY:
    utils.print_warning(
        "APIM agent-access subscription key is not present yet. "
        "This is not expected in the new sequence. "
        "Run Notebook 5 first, then rerun this cell to validate key presence."
    )
else:
    utils.print_ok(f"Subscription key present: {PF_SUB_KEY[:8]}...")

In [ ]:
# ── 2️⃣ Verify APIM gateway is reachable ─────────────────────────────────────
apim_url_out = run(
    f"az apim show -g {hub_rg} -n {apim_name} --query gatewayUrl -o tsv",
    "APIM gateway URL fetched", "Failed to get APIM gateway URL"
 )
if not apim_url_out.success:
    raise RuntimeError("Could not retrieve APIM gateway URL.")

apim_gateway_url = apim_url_out.stdout.strip() if hasattr(apim_url_out, "stdout") else apim_url_out.json_data
# Fallback if json_data is not a string
if isinstance(apim_gateway_url, dict):
    apim_gateway_url = apim_gateway_url.get("gatewayUrl", "")

# Persist APIM gateway URL into azd env for downstream runtime use.
set_azd_env("APIM_GATEWAY_URL", apim_gateway_url)
utils.print_ok(f"APIM Gateway URL: {apim_gateway_url}")

import requests
try:
    resp = requests.get(apim_gateway_url, timeout=10, verify=False)
    utils.print_ok(f"APIM gateway reachable — HTTP {resp.status_code}")
except Exception as e:
    utils.print_warning(f"Could not reach APIM gateway directly: {e}. This may be expected if APIM is on a private network.")

### 3️⃣ Persona capability matrix — review and confirm

The table below defines the authorization matrix for this workshop.  
In the scenario notebooks, you will set the persona in the message governance block.

In [ ]:
PERSONA_MATRIX = {
    "external_customer": {
        "recommendation":       {"allowed": True,  "disclaimer_required": False, "confidence_threshold": None},
        "compatibility":        {"allowed": True,  "disclaimer_required": True,  "confidence_threshold": 0.90},
        "sample_request":       {"allowed": True,  "disclaimer_required": False, "confidence_threshold": None},
        "advanced_analysis":    {"allowed": False, "reason": "Internal scientists only"},
        "out_of_domain":        {"allowed": False, "reason": "Only product-related queries allowed"},
    },
    "internal_scientist": {
        "recommendation":       {"allowed": True,  "disclaimer_required": False, "confidence_threshold": None},
        "compatibility":        {"allowed": True,  "disclaimer_required": True,  "confidence_threshold": 0.90},
        "sample_request":       {"allowed": False, "reason": "Sample requests are for external customers"},
        "advanced_analysis":    {"allowed": True,  "disclaimer_required": True,  "confidence_threshold": 0.90},
        "out_of_domain":        {"allowed": False, "reason": "Only product-related queries allowed"},
    },
}

print("=" * 65)
print("PERSONA AUTHORIZATION MATRIX")
print("=" * 65)
for persona, caps in PERSONA_MATRIX.items():
    print(f"\n  [{persona.upper()}]")
    for cap, rules in caps.items():
        icon = "✅" if rules.get("allowed") else "❌"
        disc = " (⚠️  disclaimer required)" if rules.get("disclaimer_required") else ""
        conf = f" (min confidence {rules['confidence_threshold']})" if rules.get("confidence_threshold") else ""
        reason = f" — {rules['reason']}" if "reason" in rules else ""
        print(f"    {icon} {cap}{disc}{conf}{reason}")

# Persist persona matrix to azd env for runtime-aware notebooks.
set_azd_env("PF_PERSONA_MATRIX", json.dumps(PERSONA_MATRIX))

print()
utils.print_ok("✅ Authorization and persona checks PASSED. Proceed to Notebook 7.")